# 🎯 Domain-Adaptation Fine-Tune — Speech Emotion Recognition

Loads the already-trained model from Drive and does a **short, low-LR follow-up fine-tune** targeting the specific gap found during testing: strong accuracy (89.94%) on acted-corpus test data, but poor accuracy (~29%) on real, self-recorded `.ogg` audio.

**Root causes being targeted here:**
1. **Acted vs. natural speech** — training data was scripted/exaggerated studio delivery; real audio is unscripted and subtler → fixed by adding **full MELD** (naturalistic TV dialogue, all 7 emotions, not just the surprise-only slice used before).
2. **Clean vs. noisy/compressed audio** — training data was near-silent-background WAV; real audio was `.ogg`, phone mic, room noise → fixed by **background-noise augmentation** (UrbanSound8K) and **codec-compression simulation** applied to a slice of the original training data.

**Safety first:** the existing saved model is copied to a new folder before anything is touched — the original stays completely untouched no matter what happens in this notebook.

> ⚠️ GPU runtime required (T4+). This is a short fine-tune (small dataset, few epochs) — expect minutes, not hours.


## 1. Install dependencies

In [1]:
!pip install -q transformers datasets evaluate accelerate torchaudio librosa soundfile kaggle scikit-learn audiomentations


## 2. Kaggle API setup

In [2]:
from google.colab import files
import os

print("Upload your kaggle.json file (from https://www.kaggle.com/settings)")
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json
print("Kaggle API configured.")


Upload your kaggle.json file (from https://www.kaggle.com/settings)


Saving kaggle.json to kaggle (1).json
Kaggle API configured.


## 3. Mount Drive & safely copy the existing model to a NEW folder

The original model at `MindSight_Models/audio_model` is never opened for writing in this notebook — we only ever read from the **copy**.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import shutil

BASE_MODEL_DIR = "/content/drive/MyDrive/MindSight_Models/audio_model"                       # original — untouched
DOMAIN_ADAPTED_DIR = "/content/drive/MyDrive/MindSight_Models/audio_model_domain_adapted_v1"  # new, safe working copy

assert os.path.exists(BASE_MODEL_DIR), f"Base model not found at {BASE_MODEL_DIR} - check the path."

if os.path.exists(DOMAIN_ADAPTED_DIR):
    print(f"{DOMAIN_ADAPTED_DIR} already exists - not overwriting. Delete it manually first if you want a clean copy.")
else:
    shutil.copytree(BASE_MODEL_DIR, DOMAIN_ADAPTED_DIR)
    print(f"Copied base model:\n  {BASE_MODEL_DIR}\n  -> {DOMAIN_ADAPTED_DIR}")

print("\nFiles in the new working copy:")
!ls "{DOMAIN_ADAPTED_DIR}"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/MindSight_Models/audio_model_domain_adapted_v1 already exists - not overwriting. Delete it manually first if you want a clean copy.

Files in the new working copy:
config.json  model.safetensors	preprocessor_config.json


## 4. Load the model from the SAFE COPY (not the original)

In [4]:
import torch
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

device = "cuda" if torch.cuda.is_available() else "cpu"

feature_extractor = AutoFeatureExtractor.from_pretrained(DOMAIN_ADAPTED_DIR)
model = AutoModelForAudioClassification.from_pretrained(DOMAIN_ADAPTED_DIR).to(device)

EMOTIONS = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
label2id = model.config.label2id
id2label = model.config.id2label
TARGET_SR = 16000
MAX_DURATION = 4.0

print("Loaded from safe copy. Model on:", device)
print("label2id:", label2id)


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Loaded from safe copy. Model on: cuda
label2id: {'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5, 'surprise': 6}


## 5. Download domain-adaptation datasets

- **MELD** (all 7 emotions this time, not just surprise) — naturalistic dialogue audio
- **UrbanSound8K** — real background noise, used only as an augmentation source (mixed into speech clips), never as training labels itself


In [5]:
os.makedirs('/content/data', exist_ok=True)
os.chdir('/content/data')

datasets_to_get = [
    "zaber666/meld-dataset",
    "chrisfilo/urbansound8k",
]

for ds in datasets_to_get:
    folder = ds.split("/")[-1]
    if not os.path.exists(folder):
        print(f"Downloading {ds} ...")
        !kaggle datasets download -d {ds} -p {folder} --unzip
    else:
        print(f"{folder} already exists, skipping.")

os.chdir('/content')
print("Domain-adaptation datasets ready.")


meld-dataset already exists, skipping.
urbansound8k already exists, skipping.
Domain-adaptation datasets ready.


## 6. Build the FULL MELD set (all 7 emotions, naturalistic dialogue)

Same extraction approach as before (audio pulled from MELD's video files via `ffmpeg`), but now keeping every emotion instead of just surprise.


In [6]:
import subprocess, glob, pandas as pd

MELD_ROOT = "/content/data/meld-dataset"
MELD_AUDIO_DIR = "/content/data/meld_full_audio"
os.makedirs(MELD_AUDIO_DIR, exist_ok=True)

meld_emotion_map = {
    "anger": "angry", "disgust": "disgust", "fear": "fear",
    "joy": "happy", "neutral": "neutral", "sadness": "sad", "surprise": "surprise",
}

def find_file(root, name_fragment):
    for f in glob.glob(f"{root}/**/{name_fragment}", recursive=True):
        return f
    return None

meld_rows = []

def extract_meld_split(csv_fragment, max_clips_per_emotion=400):
    csv_path = find_file(MELD_ROOT, csv_fragment)
    if csv_path is None:
        print(f"Could not find {csv_fragment} - skipping.")
        return 0

    split_df = pd.read_csv(csv_path)
    split_df["mapped_emotion"] = split_df["Emotion"].str.lower().map(meld_emotion_map)
    split_df = split_df.dropna(subset=["mapped_emotion"])

    # cap per emotion so one MELD split doesn't dominate the fine-tune set
    split_df = split_df.groupby("mapped_emotion", group_keys=False).apply(
        lambda g: g.head(max_clips_per_emotion)
    )

    added = 0
    for _, row in split_df.iterrows():
        vid_name = f"dia{row['Dialogue_ID']}_utt{row['Utterance_ID']}.mp4"
        vid_path = find_file(MELD_ROOT, vid_name)
        if vid_path is None:
            continue
        out_wav = os.path.join(MELD_AUDIO_DIR, vid_name.replace(".mp4", ".wav"))
        if not os.path.exists(out_wav):
            result = subprocess.run(
                ["ffmpeg", "-y", "-i", vid_path, "-vn", "-acodec", "pcm_s16le",
                 "-ar", "16000", "-ac", "1", out_wav],
                capture_output=True
            )
            if result.returncode != 0 or not os.path.exists(out_wav):
                continue
        meld_rows.append((out_wav, row["mapped_emotion"]))
        added += 1
    return added

n1 = extract_meld_split("train_sent_emo.csv", max_clips_per_emotion=400)
n2 = extract_meld_split("dev_sent_emo.csv", max_clips_per_emotion=100)
print(f"Extracted {n1 + n2} MELD clips across all 7 emotions ({n1} train split + {n2} dev split).")

meld_df = pd.DataFrame(meld_rows, columns=["filepath", "emotion"])
print(meld_df["emotion"].value_counts())


/tmp/ipykernel_19149/2489379145.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  split_df = split_df.groupby("mapped_emotion", group_keys=False).apply(
/tmp/ipykernel_19149/2489379145.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  split_df = split_df.groupby("mapped_emotion", group_keys=False).apply(


Extracted 3101 MELD clips across all 7 emotions (2539 train split + 562 dev split).
emotion
angry       500
neutral     500
happy       500
sad         500
surprise    500
fear        308
disgust     293
Name: count, dtype: int64


## 7. Noise + codec augmentation on a slice of the ORIGINAL training data

This is the piece that directly targets the domain gap: real audio had background noise and compression artifacts that clean acted-corpus WAVs don't have. We simulate both on a sample of your original training clips (re-downloaded here since this is a fresh notebook) so the model sees "emotion X, but noisy and compressed" during this fine-tune.


In [7]:
# Re-download the original 4 base datasets (needed as a source to augment from)
os.chdir('/content/data')
base_datasets = [
    "uwrfkaggler/ravdess-emotional-speech-audio",
    "ejlok1/cremad",
    "ejlok1/toronto-emotional-speech-set-tess",
    "ejlok1/surrey-audiovisual-expressed-emotion-savee",
]
for ds in base_datasets:
    folder = ds.split("/")[-1]
    if not os.path.exists(folder):
        print(f"Downloading {ds} ...")
        !kaggle datasets download -d {ds} -p {folder} --unzip
    else:
        print(f"{folder} already exists, skipping.")
os.chdir('/content')

import re

ravdess_map = {"01": "neutral", "02": "neutral", "03": "happy", "04": "sad",
               "05": "angry", "06": "fear", "07": "disgust", "08": "surprise"}
cremad_map = {"ANG": "angry", "DIS": "disgust", "FEA": "fear", "HAP": "happy", "NEU": "neutral", "SAD": "sad"}
tess_map = {"angry": "angry", "disgust": "disgust", "fear": "fear", "happy": "happy",
            "neutral": "neutral", "sad": "sad", "ps": "surprise", "surprise": "surprise",
            "pleasant_surprise": "surprise"}

def savee_emotion(fname):
    base = os.path.basename(fname).lower()
    prefix = re.sub(r"\d+.*", "", base)
    mapping = {"sa": "sad", "su": "surprise", "a": "angry", "d": "disgust",
               "f": "fear", "h": "happy", "n": "neutral"}
    for k, v in mapping.items():
        if prefix.startswith(k):
            return v
    return None

orig_rows = []
for f in glob.glob("/content/data/ravdess-emotional-speech-audio/**/*.wav", recursive=True):
    parts = os.path.basename(f).split("-")
    if len(parts) >= 3 and ravdess_map.get(parts[2]):
        orig_rows.append((f, ravdess_map[parts[2]]))
for f in glob.glob("/content/data/cremad/**/*.wav", recursive=True):
    parts = os.path.basename(f).split("_")
    if len(parts) >= 3 and cremad_map.get(parts[2]):
        orig_rows.append((f, cremad_map[parts[2]]))
for f in glob.glob("/content/data/toronto-emotional-speech-set-tess/**/*.wav", recursive=True):
    name = os.path.basename(f).lower()
    for key, emo in tess_map.items():
        if key in name:
            orig_rows.append((f, emo))
            break
for f in glob.glob("/content/data/surrey-audiovisual-expressed-emotion-savee/**/*.wav", recursive=True):
    emo = savee_emotion(f)
    if emo:
        orig_rows.append((f, emo))

orig_df = pd.DataFrame(orig_rows, columns=["filepath", "emotion"]).drop_duplicates(subset="filepath")
print("Original base pool available for augmentation:", len(orig_df))

# Sample a modest slice per class to noise/codec-augment (keeps this fine-tune stage small & fast)
SAMPLES_PER_CLASS = 60
sample_df = orig_df.groupby("emotion", group_keys=False).apply(
    lambda g: g.sample(min(len(g), SAMPLES_PER_CLASS), random_state=42)
)
print(f"Sampled {len(sample_df)} clips to noise/codec-augment.")


ravdess-emotional-speech-audio already exists, skipping.
cremad already exists, skipping.
toronto-emotional-speech-set-tess already exists, skipping.
Dataset URL: https://www.kaggle.com/datasets/ejlok1/surrey-audiovisual-expressed-emotion-savee
License(s): copyright-authors
100% 107M/107M [00:06<00:00, 16.1MB/s]

Original base pool available for augmentation: 10442
Sampled 420 clips to noise/codec-augment.


/tmp/ipykernel_19149/2640569533.py:62: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sample_df = orig_df.groupby("emotion", group_keys=False).apply(


In [8]:
import librosa
import soundfile as sf
from audiomentations import Compose, AddBackgroundNoise, PitchShift, Gain

NOISE_DIR = "/content/data/urbansound8k"  # audiomentations will recursively find wav files under here
AUG_OUT_DIR = "/content/data/domain_aug_audio"
os.makedirs(AUG_OUT_DIR, exist_ok=True)

domain_augmenter = Compose([
    AddBackgroundNoise(sounds_path=NOISE_DIR, min_snr_db=3.0, max_snr_db=18.0, p=0.8),
    Gain(min_gain_db=-6, max_gain_db=6, p=0.5),
    PitchShift(min_semitones=-2, max_semitones=2, p=0.3),
])

def apply_codec_compression(in_path, out_path, bitrate="24k"):
    """Round-trips audio through ogg/opus compression to simulate real-world .ogg artifacts."""
    tmp_ogg = out_path.replace(".wav", ".ogg")
    r1 = subprocess.run(["ffmpeg", "-y", "-i", in_path, "-c:a", "libopus", "-b:a", bitrate, tmp_ogg],
                         capture_output=True)
    if r1.returncode != 0:
        return False
    r2 = subprocess.run(["ffmpeg", "-y", "-i", tmp_ogg, "-ar", "16000", "-ac", "1", out_path],
                         capture_output=True)
    os.path.exists(tmp_ogg) and os.remove(tmp_ogg)
    return r2.returncode == 0 and os.path.exists(out_path)

domain_aug_rows = []
for i, (_, row) in enumerate(sample_df.iterrows()):
    try:
        speech, sr = librosa.load(row["filepath"], sr=TARGET_SR)
        noisy = domain_augmenter(samples=speech, sample_rate=TARGET_SR)
        noisy_path = os.path.join(AUG_OUT_DIR, f"noisy_{i}_{row['emotion']}.wav")
        sf.write(noisy_path, noisy, TARGET_SR)

        compressed_path = os.path.join(AUG_OUT_DIR, f"codec_{i}_{row['emotion']}.wav")
        ok = apply_codec_compression(noisy_path, compressed_path)
        final_path = compressed_path if ok else noisy_path

        domain_aug_rows.append((final_path, row["emotion"]))
    except Exception as e:
        print("  skipped:", row["filepath"], e)

domain_aug_df = pd.DataFrame(domain_aug_rows, columns=["filepath", "emotion"])
print(f"Created {len(domain_aug_df)} noise+codec-augmented domain-adaptation clips.")
print(domain_aug_df["emotion"].value_counts())


/usr/local/lib/python3.13/dist-packages/audiomentations/core/audio_loading_utils.py:36: UserWarning: /content/data/urbansound8k/fold1/180937-7-1-11.wav had to be resampled from 96000 Hz to 16000 Hz. This hurt execution time.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/audiomentations/core/audio_loading_utils.py:36: UserWarning: /content/data/urbansound8k/fold2/175844-1-0-0.wav had to be resampled from 48000 Hz to 16000 Hz. This hurt execution time.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/audiomentations/core/audio_loading_utils.py:36: UserWarning: /content/data/urbansound8k/fold6/111386-5-1-7.wav had to be resampled from 44100 Hz to 16000 Hz. This hurt execution time.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/audiomentations/core/audio_loading_utils.py:36: UserWarning: /content/data/urbansound8k/fold3/13230-0-0-24.wav had to be resampled from 44100 Hz to 16000 Hz. This hurt execution time.
  warnings.warn(
/usr/local/lib/python3.13/dist-pack

Created 420 noise+codec-augmented domain-adaptation clips.
emotion
angry       60
disgust     60
fear        60
happy       60
neutral     60
sad         60
surprise    60
Name: count, dtype: int64


## 8. Combine into the fine-tuning dataset & split

In [9]:
from sklearn.model_selection import train_test_split

ft_df = pd.concat([meld_df, domain_aug_df], ignore_index=True).drop_duplicates(subset="filepath")
ft_df["label"] = ft_df["emotion"].map(label2id)
ft_df = ft_df.dropna(subset=["label"]).reset_index(drop=True)
ft_df["label"] = ft_df["label"].astype(int)

print("Total fine-tuning samples:", len(ft_df))
print(ft_df["emotion"].value_counts())

ft_train_df, ft_test_df = train_test_split(ft_df, test_size=0.15, stratify=ft_df["label"], random_state=42)
print(f"\nFine-tune train: {len(ft_train_df)} | Fine-tune test: {len(ft_test_df)}")


Total fine-tuning samples: 3192
emotion
sad         510
happy       502
surprise    502
angry       500
neutral     490
fear        345
disgust     343
Name: count, dtype: int64

Fine-tune train: 2713 | Fine-tune test: 479


## 9. Build HuggingFace Dataset + preprocess

In [10]:
from datasets import Dataset, Audio, DatasetDict

def to_hf_dataset(d):
    ds = Dataset.from_pandas(d[["filepath", "label"]].reset_index(drop=True))
    ds = ds.cast_column("filepath", Audio(sampling_rate=TARGET_SR))
    return ds

ft_dataset = DatasetDict({
    "train": to_hf_dataset(ft_train_df),
    "test": to_hf_dataset(ft_test_df),
})

def preprocess(batch):
    audio_arrays = [x["array"] for x in batch["filepath"]]
    inputs = feature_extractor(
        audio_arrays, sampling_rate=TARGET_SR,
        max_length=int(TARGET_SR * MAX_DURATION),
        truncation=True, padding="max_length",
    )
    batch["input_values"] = inputs["input_values"]
    return batch

import datasets.config
datasets.config.TORCHVISION_AVAILABLE = False  # avoids the unrelated datasets/torchvision import bug

ft_dataset = ft_dataset.map(preprocess, batched=True, batch_size=4, writer_batch_size=100,
                             remove_columns=["filepath"])
ft_dataset.set_format(type="torch", columns=["input_values", "label"])
print(ft_dataset)


Map:   0%|          | 0/2713 [00:00<?, ? examples/s]

Map:   0%|          | 0/479 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_values'],
        num_rows: 2713
    })
    test: Dataset({
        features: ['label', 'input_values'],
        num_rows: 479
    })
})


## 10. Focal Loss (same as before) + low-LR training config

**Key differences from the original training run** — this is a nudge, not a retrain:
- Learning rate ~5-6x lower (`5e-6` vs `3e-5`)
- Far fewer epochs (5 vs 8) on a much smaller dataset
- `eval_strategy="no"` — same OOM lesson as before; evaluate manually afterward instead


In [11]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import Trainer, TrainingArguments
from datetime import datetime

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
    def forward(self, logits, labels):
        ce_loss = F.cross_entropy(logits, labels, reduction="none")
        pt = torch.exp(-ce_loss)
        return (((1 - pt) ** self.gamma) * ce_loss).mean()

class FocalLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = FocalLoss(gamma=2.0)(logits, labels)
        return (loss, outputs) if return_outputs else loss

RUN_TAG = "domain_adapt_v1"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M")
RUN_NAME = f"{RUN_TAG}_{RUN_ID}"
OUTPUT_DIR = f"/content/{RUN_NAME}"
print("This run's unique name:", RUN_NAME)

model.freeze_feature_encoder()
model.gradient_checkpointing_enable()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    eval_strategy="no",
    save_strategy="epoch",
    num_train_epochs=5,
    learning_rate=5e-6,
    warmup_steps=20,
    weight_decay=0.01,
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    save_total_limit=1,
    report_to="none",
)

trainer = FocalLossTrainer(
    model=model,
    args=training_args,
    train_dataset=ft_dataset["train"],
    processing_class=feature_extractor,
)

trainer.train()


This run's unique name: domain_adapt_v1_20260825_0753


Step,Training Loss
10,36.978760
20,30.308072
30,19.810201
40,13.454985
50,12.325370
60,10.669336
70,10.491274
80,7.898866
90,9.381660
100,9.463869


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=850, training_loss=7.248352005902459, metrics={'train_runtime': 1471.0629, 'train_samples_per_second': 9.221, 'train_steps_per_second': 0.578, 'total_flos': 1.64448400717824e+18, 'train_loss': 7.248352005902459, 'epoch': 5.0})

## 11. Manual evaluation (batch-by-batch, avoids the earlier OOM pattern)

In [12]:
import gc

del trainer
gc.collect()
torch.cuda.empty_cache()

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for i in range(0, len(ft_dataset["test"]), 4):
        batch = ft_dataset["test"][i:i+4]
        input_values = torch.tensor(batch["input_values"]).to(device)
        logits = model(input_values).logits
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(batch["label"])
        del input_values, logits
        if i % 40 == 0:
            gc.collect()
            torch.cuda.empty_cache()

from sklearn.metrics import classification_report, accuracy_score
acc = accuracy_score(all_labels, all_preds)
print(f"Domain-adaptation held-out test accuracy: {acc*100:.2f}%\n")
print(classification_report(all_labels, all_preds, target_names=EMOTIONS))


/tmp/ipykernel_19149/2876120402.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_values = torch.tensor(batch["input_values"]).to(device)


Domain-adaptation held-out test accuracy: 24.63%

              precision    recall  f1-score   support

       angry       0.27      0.37      0.32        75
     disgust       0.23      0.18      0.20        51
        fear       0.50      0.12      0.19        52
       happy       0.25      0.20      0.22        75
     neutral       0.24      0.35      0.28        74
         sad       0.32      0.16      0.21        77
    surprise       0.19      0.29      0.23        75

    accuracy                           0.25       479
   macro avg       0.28      0.24      0.23       479
weighted avg       0.28      0.25      0.24       479



## 12. Save the domain-adapted model (new, distinct name — original stays untouched)

In [13]:
SAVE_DIR = f"/content/drive/MyDrive/MindSight_Models/{RUN_NAME}_final"
os.makedirs(SAVE_DIR, exist_ok=True)

model.save_pretrained(SAVE_DIR)
feature_extractor.save_pretrained(SAVE_DIR)
print(f"Domain-adapted model saved to: {SAVE_DIR}")
print(f"Original model remains untouched at: {BASE_MODEL_DIR}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Domain-adapted model saved to: /content/drive/MyDrive/MindSight_Models/domain_adapt_v1_20260825_0753_final
Original model remains untouched at: /content/drive/MyDrive/MindSight_Models/audio_model


## 13. Test with sample inputs

Re-run the exact same 7 real-world clips from before (the ones that scored 2/7) to directly compare before vs. after.


In [51]:
all_results = {}


In [53]:
print("Upload test clips for this emotion:")
uploaded_tests = files.upload()

for fname in uploaded_tests.keys():
    true_label = fname.split("_")[0].lower()
    pred_label, probs = predict_emotion(fname)
    all_results[fname] = {
        "file": fname,
        "true_label": true_label,
        "predicted": pred_label,
        "correct": true_label == pred_label,
    }

Upload test clips for this emotion:


Saving angry_1.ogg to angry_1 (4).ogg
Saving angry_4.ogg to angry_4 (4).ogg
Saving angry_10.ogg to angry_10 (4).ogg
angry_1 (4).ogg      -> angry      (69.2%)
angry_4 (4).ogg      -> angry      (49.2%)   [low confidence, runner-up: fear 18.7%]
angry_10 (4).ogg     -> angry      (67.0%)


In [54]:
print("Upload test clips for this emotion:")
uploaded_tests = files.upload()

for fname in uploaded_tests.keys():
    true_label = fname.split("_")[0].lower()
    pred_label, probs = predict_emotion(fname)
    all_results[fname] = {
        "file": fname,
        "true_label": true_label,
        "predicted": pred_label,
        "correct": true_label == pred_label,
    }

Upload test clips for this emotion:


Saving disgust_7.ogg to disgust_7 (1).ogg
Saving disgust_9.ogg to disgust_9 (1).ogg
Saving disgust_10.ogg to disgust_10 (1).ogg
disgust_7 (1).ogg    -> angry      (29.4%)   [low confidence, runner-up: happy 28.1%]
disgust_9 (1).ogg    -> disgust    (62.1%)
disgust_10 (1).ogg   -> angry      (25.7%)   [low confidence, runner-up: happy 19.6%]


In [55]:
print("Upload test clips for this emotion:")
uploaded_tests = files.upload()

for fname in uploaded_tests.keys():
    true_label = fname.split("_")[0].lower()
    pred_label, probs = predict_emotion(fname)
    all_results[fname] = {
        "file": fname,
        "true_label": true_label,
        "predicted": pred_label,
        "correct": true_label == pred_label,
    }

Upload test clips for this emotion:


Saving fear_1.ogg to fear_1 (5).ogg
Saving fear_3.ogg to fear_3 (4).ogg
Saving fear_7.ogg to fear_7 (2).ogg
fear_1 (5).ogg       -> fear       (34.0%)   [low confidence, runner-up: sad 31.5%]
fear_3 (4).ogg       -> fear       (41.2%)   [low confidence, runner-up: sad 25.3%]
fear_7 (2).ogg       -> sad        (43.2%)   [low confidence, runner-up: fear 31.3%]


In [56]:
print("Upload test clips for this emotion:")
uploaded_tests = files.upload()

for fname in uploaded_tests.keys():
    true_label = fname.split("_")[0].lower()
    pred_label, probs = predict_emotion(fname)
    all_results[fname] = {
        "file": fname,
        "true_label": true_label,
        "predicted": pred_label,
        "correct": true_label == pred_label,
    }

Upload test clips for this emotion:


Saving happy_2.ogg to happy_2 (4).ogg
Saving happy_9.ogg to happy_9 (1).ogg
Saving happy_10.ogg to happy_10 (1).ogg
happy_2 (4).ogg      -> happy      (63.2%)
happy_9 (1).ogg      -> neutral    (45.2%)   [low confidence, runner-up: happy 16.9%]
happy_10 (1).ogg     -> happy      (70.5%)


In [59]:
print("Upload test clips for this emotion:")
uploaded_tests = files.upload()

for fname in uploaded_tests.keys():
    true_label = fname.split("_")[0].lower()
    pred_label, probs = predict_emotion(fname)
    all_results[fname] = {
        "file": fname,
        "true_label": true_label,
        "predicted": pred_label,
        "correct": true_label == pred_label,
    }

Upload test clips for this emotion:


Saving neutral_2.ogg to neutral_2 (2).ogg
Saving neutral_5.ogg to neutral_5 (2).ogg
Saving neutral_8.ogg to neutral_8 (1).ogg
neutral_2 (2).ogg    -> neutral    (44.4%)   [low confidence, runner-up: sad 37.8%]
neutral_5 (2).ogg    -> sad        (43.9%)   [low confidence, runner-up: neutral 39.5%]
neutral_8 (1).ogg    -> neutral    (34.8%)   [low confidence, runner-up: sad 25.1%]


In [60]:
print("Upload test clips for this emotion:")
uploaded_tests = files.upload()

for fname in uploaded_tests.keys():
    true_label = fname.split("_")[0].lower()
    pred_label, probs = predict_emotion(fname)
    all_results[fname] = {
        "file": fname,
        "true_label": true_label,
        "predicted": pred_label,
        "correct": true_label == pred_label,
    }

Upload test clips for this emotion:


Saving sad_2.ogg to sad_2 (4).ogg
Saving sad_5.ogg to sad_5 (3).ogg
Saving sad_9.ogg to sad_9 (2).ogg
sad_2 (4).ogg        -> sad        (63.7%)
sad_5 (3).ogg        -> sad        (49.9%)   [low confidence, runner-up: fear 33.1%]
sad_9 (2).ogg        -> sad        (55.7%)   [low confidence, runner-up: neutral 30.5%]


In [61]:
print("Upload test clips for this emotion:")
uploaded_tests = files.upload()

for fname in uploaded_tests.keys():
    true_label = fname.split("_")[0].lower()
    pred_label, probs = predict_emotion(fname)
    all_results[fname] = {
        "file": fname,
        "true_label": true_label,
        "predicted": pred_label,
        "correct": true_label == pred_label,
    }

Upload test clips for this emotion:


Saving surprise_2.ogg to surprise_2 (1).ogg
Saving surprise_6.ogg to surprise_6 (2).ogg
Saving surprise_7.ogg to surprise_7 (2).ogg
surprise_2 (1).ogg   -> happy      (37.9%)   [low confidence, runner-up: fear 17.9%]
surprise_6 (2).ogg   -> happy      (46.2%)   [low confidence, runner-up: surprise 23.1%]
surprise_7 (2).ogg   -> happy      (47.0%)   [low confidence, runner-up: angry 22.3%]


In [63]:
import pandas as pd

results_df = pd.DataFrame(list(all_results.values()))
correct = results_df["correct"].sum()
total = len(results_df)
accuracy = correct / total * 100 if total else 0

print(f"Overall Accuracy: {correct}/{total} correct = {accuracy:.2f}%\n")

print("Per-emotion breakdown:")
print(results_df.groupby("true_label")["correct"].agg(total="count", correct="sum"))

print("\nFull results:")
display(results_df)

Overall Accuracy: 16/28 correct = 57.14%

Per-emotion breakdown:
            total  correct
true_label                
angry           3        3
disgust         3        1
fear            3        2
happy           6        4
neutral         7        3
sad             3        3
surprise        3        0

Full results:


,file,true_label,predicted,correct
0,angry_1 (4).ogg,angry,angry,True
1,angry_4 (4).ogg,angry,angry,True
2,angry_10 (4).ogg,angry,angry,True
3,disgust_7 (1).ogg,disgust,angry,False
4,disgust_9 (1).ogg,disgust,disgust,True
5,disgust_10 (1).ogg,disgust,angry,False
6,fear_1 (5).ogg,fear,fear,True
7,fear_3 (4).ogg,fear,fear,True
8,fear_7 (2).ogg,fear,sad,False
9,happy_2 (4).ogg,happy,happy,True


In [64]:
from google.colab import drive
drive.mount('/content/drive')  # safe to call again if already mounted

from datetime import datetime
import os

RUN_TAG = "domain_adapt_final_realaudio57pct"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M")
SAVE_DIR = f"/content/drive/MyDrive/MindSight_Models/{RUN_TAG}_{RUN_ID}"

os.makedirs(SAVE_DIR, exist_ok=True)

model.save_pretrained(SAVE_DIR)
feature_extractor.save_pretrained(SAVE_DIR)

print(f"Model saved to: {SAVE_DIR}")
!ls "{SAVE_DIR}"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/drive/MyDrive/MindSight_Models/domain_adapt_final_realaudio57pct_20260825_0934
config.json  model.safetensors	preprocessor_config.json


In [65]:
!ls -la "{SAVE_DIR}"


total 1233268
-rw------- 1 root root       2038 Aug 25 09:34 config.json
-rw------- 1 root root 1262863736 Aug 25 09:34 model.safetensors
-rw------- 1 root root        212 Aug 25 09:34 preprocessor_config.json


## 14. If results still need work

- **Check the class-count printout in step 8** — if any emotion has very few combined MELD + augmented samples, that class won't improve much from this pass; consider raising `max_clips_per_emotion` in step 6 or `SAMPLES_PER_CLASS` in step 7.
- **Try 2-3 more epochs** if the model still looks under-adapted — but watch for the original acted-corpus test accuracy dropping (catastrophic forgetting); if that happens, lower the LR further instead of adding epochs.
- **The single highest-leverage next step is still real self-recorded data** — even 15-20 clips per class recorded through your actual chatbot's mic/pipeline will outperform synthetic noise/codec simulation, since it captures things simulation can't (real room acoustics, real speaking style under real conditions).
